# Hotellbokning med Prioritetsmedlemsmiddleware

Denna anteckningsbok visar **funktionsbaserad middleware** med Microsoft Agent Framework. Vi bygger vidare på exemplet med villkorligt arbetsflöde genom att lägga till ett middleware-lager som ger prioriterade medlemmar särskilda privilegier.

## Vad du kommer att lära dig:
1. **Funktionsbaserad Middleware**: Avlyssna och modifiera funktionsresultat
2. **Kontekståtkomst**: Läsa och ändra `context.result` efter körning
3. **Implementering av affärslogik**: Förmåner för prioriterade medlemmar
4. **Resultatöverskrivning**: Ändra funktionsutfall baserat på användarstatus
5. **Samma arbetsflöde, olika utfall**: Middlewarestyrda beteendeförändringar

## Arbetsflödesarkitektur med Middleware:

```
User Input: "I want to book a hotel in Paris"
                    ↓
        [availability_agent]
        - Calls hotel_booking tool
        - 🌟 priority_check middleware intercepts
        - Checks user membership status
        - IF priority + no rooms → Override to available!
        - Returns BookingCheckResult
                    ↓
        Conditional Routing
           /                    \
    [has_availability]    [no_availability]
          ↓                      ↓
    [booking_agent]        [alternative_agent]
    (Priority override!)   (Regular users)
          ↓                      ↓
       [display_result executor]
```

## Nyckelskillnad från villkorligt arbetsflöde:

**Utan Middleware** (14-conditional-workflow.ipynb):
- Paris har inga rum → Omdirigera till alternative_agent

**Med Middleware** (denna anteckningsbok):
- Vanlig användare + Paris → Inga rum → Omdirigera till alternative_agent
- Prioriterad användare + Paris → 🌟 Middleware överskrider! → Finns tillgängligt → Omdirigera till booking_agent

## Förkunskaper:
- Microsoft Agent Framework installerat
- Förståelse för villkorliga arbetsflöden (se 14-conditional-workflow.ipynb)
- GitHub-token eller OpenAI API-nyckel
- Grundläggande förståelse för middleware-mönster


In [ ]:
import asyncio
import json
import os
from collections.abc import Awaitable, Callable
from typing import Annotated, Any, Never

from agent_framework import (
    AgentExecutor,
    AgentExecutorRequest,
    AgentExecutorResponse,
    FunctionInvocationContext,
    Message,
    WorkflowBuilder,
    WorkflowContext,
    executor,
    tool,
)
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential
from dotenv import load_dotenv
from IPython.display import HTML, display
from pydantic import BaseModel

print("✅ All imports successful!")

## Steg 1: Definiera Pydantic-modeller för strukturerade utdata

Dessa modeller definierar **schemat** som agenterna kommer att returnera. Vi har lagt till ett fält `priority_override` för att spåra när mellanprogram modifierar tillgänglighetsresultatet.


In [ ]:
class BookingCheckResult(BaseModel):
    """Result from checking hotel availability at a destination."""

    destination: str
    has_availability: bool
    message: str
    # Tracks if middleware overrode the result. The Azure structured-output
    # contract requires every property to be in the JSON schema's `required`
    # array, so we cannot give this a default value the way the original
    # notebook did.
    priority_override: bool


class AlternativeResult(BaseModel):
    """Suggested alternative destination when no rooms available."""

    alternative_destination: str
    reason: str


class BookingConfirmation(BaseModel):
    """Booking suggestion when rooms are available."""

    destination: str
    action: str
    message: str


print("✅ Pydantic models defined:")
print("   - BookingCheckResult (availability check with priority_override)")
print("   - AlternativeResult (alternative suggestion)")
print("   - BookingConfirmation (booking confirmation)")

## Steg 2: Definiera Prioritetsmedlemsdatabas

För denna demo kommer vi att simulera en prioritetsmedlemsdatabas. I produktion skulle detta fråga en riktig databas eller API.

**Prioritetsmedlemmar:**
- `alice@example.com` - VIP-medlem
- `bob@example.com` - Premium-medlem  
- `priority_user` - Testkonto


In [ ]:
# Simulated priority members database
PRIORITY_MEMBERS = {
    "alice@example.com",
    "bob@example.com",
    "priority_user",
}

# Global variable to track current user (in real app, use proper session management)
current_user_id = "regular_user"  # Default: regular user


def set_user(user_id: str):
    """Set the current user for the session."""
    global current_user_id
    current_user_id = user_id
    is_priority = user_id in PRIORITY_MEMBERS
    status = "🌟 PRIORITY MEMBER" if is_priority else "👤 Regular User"

    display(
        HTML(f"""
        <div style='padding: 15px; background: {"linear-gradient(135deg, #FFD700 0%, #FFA500 100%)" if is_priority else "#e3f2fd"}; 
                    border-left: 4px solid {"#FF6B35" if is_priority else "#2196f3"}; border-radius: 4px; margin: 10px 0;'>
            <strong>👤 Current User Set:</strong> {user_id}<br>
            <strong>Status:</strong> {status}
        </div>
    """)
    )


print("✅ Priority members database created")
print(f"   Priority members: {len(PRIORITY_MEMBERS)} users")

## Steg 3: Skapa hotellbokningsverktyget

Samma som det villkorliga arbetsflödet, men nu kommer det att fångas upp av middleware!


In [ ]:
@tool(description="Check hotel room availability for a destination city")
def hotel_booking(destination: Annotated[str, "The destination city to check for hotel rooms"]) -> str:
    """
    Simulates checking hotel room availability.

    Returns JSON string with availability status.
    """
    display(
        HTML(f"""
        <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
            <strong>🔍 Tool Invoked:</strong> hotel_booking("{destination}")
        </div>
    """)
    )

    # Simulate availability check
    cities_with_rooms = ["stockholm", "seattle", "tokyo", "london", "amsterdam"]
    has_rooms = destination.lower() in cities_with_rooms

    result = {"has_availability": has_rooms, "destination": destination}

    return json.dumps(result)


print("✅ hotel_booking tool created with @tool decorator")

## Steg 4: 🌟 Skapa Prioritetskontroll-middleware (DET VIKTIGASTE FUNKTIONEN!)

Detta är **kärnfunktionaliteten** i denna anteckningsbok. Middlewaren:

1. **Avlyssnar** anropet till funktionen hotel_booking
2. **Utför** funktionen normalt genom att anropa `next(context)`
3. **Inspekterar** resultatet i `context.result`
4. **Överskriver** resultatet om användaren är prioriterad och inga rum finns tillgängliga
5. **Returnerar** det modifierade resultatet tillbaka till agenten

**Nyckelmönster:**
```python
async def my_middleware(context, next):
    await next(context)  # Kör funktionen
    # Nu innehåller context.result funktionens utdata
    if some_condition:
        context.result = new_value  # Åsidosätt!
```


In [ ]:
async def priority_check_middleware(
    context: FunctionInvocationContext,
    next: Callable[[FunctionInvocationContext], Awaitable[None]],
) -> None:
    """
    Function middleware that overrides hotel_booking results for priority members.
    
    Workflow:
    1. Let the function execute normally
    2. Check if user is a priority member
    3. If priority + no availability → Override to make rooms available!
    4. Agent will then route to booking path instead of alternative path
    """
    function_name = context.function.name

    display(
        HTML(f"""
        <div style='padding: 12px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
            <strong>🔄 Middleware:</strong> Intercepting {function_name}...
        </div>
    """)
    )

    # Execute the original function
    await next(context)

    # Now inspect and potentially modify the result
    if context.result and function_name == "hotel_booking":
        result_data = json.loads(context.result)
        destination = result_data.get("destination", "")
        has_availability = result_data.get("has_availability", False)

        # Check if user is priority member
        is_priority = current_user_id in PRIORITY_MEMBERS

        # Override logic: Priority member + no availability → Make available!
        if is_priority and not has_availability:
            display(
                HTML(f"""
                <div style='padding: 20px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); 
                            border-radius: 8px; margin: 10px 0; box-shadow: 0 4px 12px rgba(255,165,0,0.4);'>
                    <h3 style='margin: 0 0 10px 0; color: #333;'>🌟 PRIORITY OVERRIDE ACTIVATED! 🌟</h3>
                    <p style='margin: 0; color: #555; line-height: 1.6;'>
                        <strong>User:</strong> {current_user_id}<br>
                        <strong>Status:</strong> VIP Priority Member<br>
                        <strong>Action:</strong> Overriding "No Availability" for {destination}<br>
                        <strong>Result:</strong> ✅ Rooms now available for priority booking!
                    </p>
                </div>
            """)
            )

            # Override the result!
            result_data["has_availability"] = True
            result_data["priority_override"] = True
            context.result = json.dumps(result_data)

        elif not has_availability:
            display(
                HTML(f"""
                <div style='padding: 12px; background: #ffebee; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'>
                    <strong>ℹ️ Middleware:</strong> No priority override (user: {current_user_id})
                </div>
            """)
            )


print("✅ priority_check_middleware created")
print("   - Intercepts hotel_booking function")
print("   - Overrides availability for priority members")

## Steg 5: Definiera Villkorsfunktioner för Routing

Samma villkorsfunktioner som i det villkorade arbetsflödet - de undersöker den strukturerade utdata för att avgöra routing.


In [ ]:
def has_availability_condition(message: Any) -> bool:
    """Condition for routing when hotels ARE available (including priority overrides!)."""
    if not isinstance(message, AgentExecutorResponse):
        return True

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)

        # Check if this was a priority override
        override_indicator = " 🌟" if result.priority_override else ""

        display(
            HTML(f"""
            <div style='padding: 12px; background: #c8e6c9; border-left: 4px solid #4caf50; border-radius: 4px; margin: 10px 0;'>
                <strong>✅ Condition Check:</strong> has_availability = <strong>{result.has_availability}</strong> for {result.destination}{override_indicator}
            </div>
        """)
        )

        return result.has_availability
    except Exception as e:
        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffcdd2; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'>
                <strong>⚠️  Error:</strong> {str(e)}
            </div>
        """)
        )
        return False


def no_availability_condition(message: Any) -> bool:
    """Condition for routing when hotels are NOT available."""
    if not isinstance(message, AgentExecutorResponse):
        return False

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)

        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffecb3; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
                <strong>❌ Condition Check:</strong> no_availability for {result.destination}
            </div>
        """)
        )

        return not result.has_availability
    except Exception:
        return False


print("✅ Condition functions defined")

## Steg 6: Skapa egen Display Executor

Samma executor som tidigare - visar det slutgiltiga arbetsflödets output.


In [ ]:
@executor(id="display_result")
async def display_result(response: AgentExecutorResponse, ctx: WorkflowContext[Never, str]) -> None:
    """Display the final result as workflow output."""
    display(
        HTML("""
        <div style='padding: 15px; background: #f3e5f5; border-left: 4px solid #9c27b0; border-radius: 4px; margin: 10px 0;'>
            <strong>📤 Display Executor:</strong> Yielding workflow output
        </div>
    """)
    )

    await ctx.yield_output(response.agent_run_response.text)


print("✅ display_result executor created")

## Steg 7: Ladda miljövariabler

Konfigurera LLM-klienten (GitHub Models eller OpenAI).


In [ ]:
# Load environment variables
load_dotenv()

# Configure the Azure AI Foundry provider with keyless authentication
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Steg 8: Skapa AI-agenter med Middleware

**VIKTIG SKILLNAD:** När vi skapar availability_agent, skickar vi med parametern `middleware`!

Så här injicerar vi priority_check_middleware i agentens funktionsanropspipeline.


In [ ]:
# Agent 1: Check availability with tool + middleware
availability_agent = AgentExecutor(
    await provider.create_agent(
        name="availability-agent",
        instructions=(
            "You are a hotel booking assistant that checks room availability. "
            "Use the hotel_booking tool to check if rooms are available at the destination. "
            "Return JSON with fields: destination (string), has_availability (bool), message (string), "
            "and priority_override (bool, true if priority member got special access). "
            "The message should summarize the availability status and mention if priority override occurred."
        ),
        tools=[hotel_booking],
        default_options={"response_format": BookingCheckResult},
        middleware=[priority_check_middleware],  # 🌟 MIDDLEWARE INJECTION!
    ),
    id="availability_agent",
)

# Agent 2: Suggest alternative (when no rooms)
alternative_agent = AgentExecutor(
    await provider.create_agent(
        name="alternative-agent",
        instructions=(
            "You are a helpful travel assistant. When a user cannot find hotels in their requested city, "
            "suggest an alternative nearby city that has availability. "
            "Return JSON with fields: alternative_destination (string) and reason (string). "
            "Make your suggestion sound appealing and helpful."
        ),
        default_options={"response_format": AlternativeResult},
    ),
    id="alternative_agent",
)

# Agent 3: Suggest booking (when rooms available)
booking_agent = AgentExecutor(
    await provider.create_agent(
        name="booking-agent",
        instructions=(
            "You are a booking assistant. The user has found available hotel rooms. "
            "Encourage them to book by highlighting the destination's appeal. "
            "If priority_override is true in the input, mention that they received priority member access. "
            "Return JSON with fields: destination (string), action (string), and message (string). "
            "The action should be 'book_now' and message should be encouraging."
        ),
        default_options={"response_format": BookingConfirmation},
    ),
    id="booking_agent",
)

display(
    HTML("""
    <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
        <strong>✅ Created 3 Agents:</strong>
        <ul style='margin: 10px 0 0 0;'>
            <li><strong>availability_agent</strong> - WITH priority_check_middleware 🌟</li>
            <li><strong>alternative_agent</strong> - Suggests alternative cities</li>
            <li><strong>booking_agent</strong> - Encourages booking</li>
        </ul>
    </div>
""")
)

## Steg 9: Bygg arbetsflödet

Samma arbetsflödesstruktur som tidigare - villkorlig dirigering baserat på tillgänglighet.


In [ ]:
# Build the workflow with conditional routing
workflow = (
    WorkflowBuilder(
        start_executor=availability_agent,
        output_executors=[display_result],
    )
    # NO AVAILABILITY PATH
    .add_edge(availability_agent, alternative_agent, condition=no_availability_condition)
    .add_edge(alternative_agent, display_result)
    # HAS AVAILABILITY PATH (can be triggered by middleware override!)
    .add_edge(availability_agent, booking_agent, condition=has_availability_condition)
    .add_edge(booking_agent, display_result)
    .build()
)

display(
    HTML("""
    <div style='padding: 20px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; border-radius: 8px; margin: 10px 0;'>
        <h3 style='margin: 0 0 15px 0;'>✅ Workflow Built Successfully!</h3>
        <p style='margin: 0; line-height: 1.6;'>
            <strong>Conditional Routing (Middleware-Aware):</strong><br>
            • If <strong>NO availability</strong> → alternative_agent → display_result<br>
            • If <strong>availability</strong> (or 🌟 <strong>priority override</strong>) → booking_agent → display_result
        </p>
    </div>
""")
)

## Steg 10: Testfall 1 - Vanlig användare i Paris (Ingen överskrivning)

En vanlig användare försöker boka Paris → Inga rum → Rutas till alternative_agent


In [ ]:
# Set as regular user
set_user("regular_user")

display(
    HTML("""
    <div style='padding: 20px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #e65100;'>🧪 TEST CASE 1: Regular User + Paris</h3>
        <p style='margin: 0;'><strong>Expected:</strong> No rooms → No middleware override → Alternative suggestion</p>
    </div>
""")
)

# Create request
request_regular = AgentExecutorRequest(
    messages=[Message(role="user", text="I want to book a hotel in Paris")], should_respond=True
)

# Run workflow
events_regular = await workflow.run(request_regular)
outputs_regular = events_regular.get_outputs()

# Display results
if outputs_regular:
    result_regular = AlternativeResult.model_validate_json(outputs_regular[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: #fff; border: 2px solid #ff9800; border-radius: 12px; margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0; color: #e65100;'>📊 RESULT (Regular User)</h3>
            <div style='background: #fff3e0; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0;'><strong>Status:</strong> ❌ No rooms in Paris</p>
                <p style='margin: 0 0 10px 0;'><strong>Middleware:</strong> No priority override (regular user)</p>
                <p style='margin: 0 0 10px 0;'><strong>Alternative:</strong> 🏨 {result_regular.alternative_destination}</p>
                <p style='margin: 0;'><strong>Reason:</strong> {result_regular.reason}</p>
            </div>
        </div>
    """)
    )

## Steg 11: Testfall 2 - 🌟 Prioriterad användare i Paris (MED Override!)

En prioriterad medlem försöker boka Paris → Inga rum tillgängliga initialt → 🌟 Middleware åsidosätter! → Dirigerar till booking_agent

**Detta är den centrala demonstrationen av middleware-kraft!**


In [ ]:
# Set as priority user
set_user("priority_user")

display(
    HTML("""
    <div style='padding: 20px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #333;'>🧪 TEST CASE 2: 🌟 Priority User + Paris</h3>
        <p style='margin: 0; color: #555;'><strong>Expected:</strong> No rooms → 🌟 MIDDLEWARE OVERRIDE → Rooms available → Booking suggestion!</p>
    </div>
""")
)

# Create request
request_priority = AgentExecutorRequest(
    messages=[Message(role="user", text="I want to book a hotel in Paris")], should_respond=True
)

# Run workflow
events_priority = await workflow.run(request_priority)
outputs_priority = events_priority.get_outputs()

# Display results
if outputs_priority:
    result_priority = BookingConfirmation.model_validate_json(outputs_priority[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); border-radius: 12px;
                    box-shadow: 0 8px 16px rgba(255,165,0,0.4); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0; color: #333;'>🏆 RESULT (Priority Member) 🌟</h3>
            <div style='background: white; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ✅ Rooms Available (Priority Override!)</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Middleware:</strong> 🌟 OVERRIDE ACTIVATED!</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Destination:</strong> 🏨 {result_priority.destination}</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Action:</strong> {result_priority.action}</p>
                <p style='margin: 0; font-size: 14px; color: #666;'><strong>Message:</strong> {result_priority.message}</p>
                <div style='margin-top: 15px; padding: 15px; background: #fff3cd; border-radius: 6px; border-left: 4px solid #FF6B35;'>
                    <strong>💡 What Just Happened:</strong><br>
                    1. hotel_booking tool returned "no availability"<br>
                    2. priority_check_middleware intercepted the result<br>
                    3. Middleware checked user status: priority_user ✅<br>
                    4. Middleware OVERRODE the result to "available"<br>
                    5. Workflow routed to booking_agent instead of alternative_agent!
                </div>
            </div>
        </div>
    """)
    )

## Steg 12: Testfall 3 - Prioriterad användare i Stockholm (redan tillgänglig)

Prioriterad användare försöker Stockholm → Rum tillgängliga → Ingen åsidosättning behövs → Dirigerar till booking_agent

Detta visar att middleware endast agerar när det behövs!


In [ ]:
# Priority user is still set from previous test

display(
    HTML("""
    <div style='padding: 20px; background: #e8f5e9; border-left: 4px solid #4caf50; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #1b5e20;'>🧪 TEST CASE 3: Priority User + Stockholm</h3>
        <p style='margin: 0;'><strong>Expected:</strong> Rooms available → No override needed → Booking suggestion</p>
    </div>
""")
)

# Create request
request_stockholm = AgentExecutorRequest(
    messages=[Message(role="user", text="I want to book a hotel in Stockholm")], should_respond=True
)

# Run workflow
events_stockholm = await workflow.run(request_stockholm)
outputs_stockholm = events_stockholm.get_outputs()

# Display results
if outputs_stockholm:
    result_stockholm = BookingConfirmation.model_validate_json(outputs_stockholm[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #4caf50 0%, #8bc34a 100%); color: white; border-radius: 12px;
                    box-shadow: 0 4px 12px rgba(76,175,80,0.3); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0;'>🏆 RESULT (Priority User - No Override Needed)</h3>
            <div style='background: white; color: #333; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ✅ Rooms Available (Natural)</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Middleware:</strong> No override needed</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Destination:</strong> 🏨 {result_stockholm.destination}</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Action:</strong> {result_stockholm.action}</p>
                <p style='margin: 0; font-size: 14px; color: #666;'><strong>Message:</strong> {result_stockholm.message}</p>
                <div style='margin-top: 15px; padding: 15px; background: #e8f5e9; border-radius: 6px; border-left: 4px solid #4caf50;'>
                    <strong>💡 Middleware Behavior:</strong><br>
                    • hotel_booking returned "available" naturally<br>
                    • Middleware saw available = true → No override needed<br>
                    • Workflow proceeded normally to booking_agent
                </div>
            </div>
        </div>
    """)
    )

## Viktiga insikter och middleware-koncept

### ✅ Vad du har lärt dig:

#### **1. Funktionsbaserat middleware-mönster**

Middleware avlyssnar funktionsanrop med hjälp av en enkel asynkron funktion:

```python
async def my_middleware(
    context: FunctionInvocationContext,
    next: Callable,
) -> None:
    # Före funktionsutförande
    print("Intercepting...")
    
    # Utför funktionen
    await next(context)
    
    # Efter funktionsutförande - inspektera resultatet
    if context.result:
        # Ändra resultatet om det behövs
        context.result = modified_value
```

#### **2. Åtkomst till kontext och överskrivning av resultat**

- `context.function` - Åtkomst till funktionen som anropas
- `context.arguments` - Läsa funktionsargument
- `context.kwargs` - Åtkomst till ytterligare parametrar
- `await next(context)` - Kör funktionen
- `context.result` - Läsa/ändra funktionens output

#### **3. Implementering av affärslogik**

Vårt middleware implementerar prioriterade medlemsförmåner:
- **Vanliga användare**: Inga ändringar, standardflöde
- **Prioriterade användare**: Överskrider "ingen tillgänglighet" → "tillgänglig"
- **Villkorlig logik**: Överskrider endast när det behövs

#### **4. Samma arbetsflöde, olika resultat**

Kraften i middleware:
- ✅ Inga ändringar i arbetsflödets struktur
- ✅ Inga ändringar i verktygsfunktionen
- ✅ Inga ändringar i villkorlig routinglogik
- ✅ Bara middleware → Olika beteende!

### 🚀 Verkliga tillämpningar:

1. **VIP/Premiumfunktioner**
   - Överskrid hastighetsbegränsningar för premiumanvändare
   - Ge prioriterad åtkomst till resurser
   - Lås upp premiumfunktioner dynamiskt

2. **A/B-testning**
   - Skicka användare till olika implementationer
   - Testa nya funktioner med specifika användare
   - Gradvisa funktionsutrullningar

3. **Säkerhet & efterlevnad**
   - Granska funktionsanrop
   - Blockera känsliga operationer
   - Tillämpa affärsregler

4. **Prestandaoptimering**
   - Cachar resultat för specifika användare
   - Hoppa över kostsamma operationer när möjligt
   - Dynamisk resursallokering

5. **Felsökning & återförsök**
   - Fånga och hantera fel på ett smidigt sätt
   - Implementera återförsökslogik
   - Falla tillbaka till alternativa implementationer

6. **Loggning & övervakning**
   - Spåra funktionstider
   - Logga parametrar och resultat
   - Övervaka användningsmönster

### 🔑 Viktiga skillnader från dekoratörer:

| Funktion | Dekoratör | Middleware |
|---------|-----------|------------|
| **Omfattning** | Enskild funktion | Alla funktioner i agenten |
| **Flexibilitet** | Fast vid definition | Dynamisk vid körning |
| **Kontext** | Begränsad | Full agentkontext |
| **Sammansättning** | Flera dekoratörer | Middleware-pipeline |
| **Agentmedveten** | Nej | Ja (åtkomst till agenttillstånd) |

### 📚 När man ska använda middleware:

✅ **Använd middleware när:**
- Du behöver ändra beteende baserat på användar-/sessionsstatus
- Du vill tillämpa logik på flera funktioner
- Du behöver åtkomst till agentnivå-kontext
- Du implementerar tvärgående bekymmer (loggning, autentisering etc.)

❌ **Använd inte middleware när:**
- Enkel inmatningsvalidering (använd Pydantic)
- Funktionsspecifik logik (behåll i funktionen)
- Engångsändringar (ändra bara funktionen)

### 🎓 Avancerade mönster:

```python
# Flera middleware (exekveringsordning spelar roll!)
middleware=[
    logging_middleware,      # Loggar först
    auth_middleware,         # Sedan kontrollerar auktorisering
    cache_middleware,        # Sedan kontrollerar cache
    rate_limit_middleware,   # Sedan begränsar hastighet
    priority_check_middleware  # Slutligen prioriteringskontroll
]

# Villkorlig middleware-exekvering
async def conditional_middleware(context, next):
    if should_execute(context):
        await next(context)
        # Modifiera resultat
    else:
        # Hoppa över exekvering helt
        context.result = cached_value
```

### 🔗 Relaterade koncept:

- **Agent Middleware**: Avlyssnar agent.run()-anrop
- **Funktionsmiddleware**: Avlyssnar verktygsfunktionsanrop (det vi använde!)
- **Middleware-pipeline**: Kedja av middleware som körs i ordning
- **Kontextpropagering**: Skicka tillstånd genom middleware-kedjan


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Ansvarsfriskrivning**:
Detta dokument har översatts med hjälp av AI-översättningstjänsten [Co-op Translator](https://github.com/Azure/co-op-translator). Även om vi strävar efter noggrannhet, var vänlig notera att automatiska översättningar kan innehålla fel eller brister. Det ursprungliga dokumentet på dess modersmål bör betraktas som den auktoritativa källan. För kritisk information rekommenderas professionell mänsklig översättning. Vi ansvarar inte för några missförstånd eller feltolkningar som uppstår till följd av användningen av denna översättning.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
